# Первый спринт: исследование предоставленного графика
Цель - проверить качество данных и попробовать объяснить изменение между соседними месяцами. Исходную книгу не меняем. Проверяем сохранённые значения.

График платежей предоставлен пользователем. Ставки оценки и стрессы в опыте ЭК синтетические. Даты платежей не являются датами наблюдения NPV или ЭК. Валюта отдельно не подтверждена.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/factor_analysis").exists()), None)
if ROOT is None:
    raise RuntimeError("Откройте ноутбук внутри репозитория factor-analysis")
sys.path.insert(0, str(ROOT / "src"))
from factor_analysis.workbook import inspect_annuity, read_workbook
from factor_analysis.profiling import profile_table
from factor_analysis.sprint1_demo import run_schedule_change, run_cashflow_demo
# После переноса замените на свой путь к XLSX.
SOURCE = Path("/Users/andrey/Downloads/annuity-payment-schedule (1).xlsx")
if not SOURCE.is_file():
    raise FileNotFoundError("Укажите в SOURCE путь к исходному XLSX")
print("Источник:", SOURCE.name)

Источник: annuity-payment-schedule (1).xlsx


## 1. Структура и происхождение
Проверяем ошибки Excel и формулы без сохранённых значений. Аналитическая таблица содержит платежи без заголовков и итогов. Это проверка сохранённой книги, не пересчёт всех формул.

In [2]:
sheets = read_workbook(SOURCE)
for name, values in sheets.items():
    formulas = [c for c in values.values() if c["formula"] is not None]
    print(name, "| ячеек:", len(values), "| формул:", len(formulas),
          "| формул без кэша:", sum(c["value"] is None for c in formulas),
          "| ошибок Excel:", sum(c["error"] for c in values.values()))
audit = inspect_annuity(SOURCE)
frame = pd.DataFrame(audit["cashflows"])
print("SHA-256:", audit["source"]["sha256"])
print("Количество платежей:", len(frame))
print(frame.head().to_string(index=False))

График платежей | ячеек: 5481 | формул: 3013 | формул без кэша: 1330 | ошибок Excel: 0
Расчет платежа | ячеек: 3045 | формул: 3006 | формул без кэша: 1330 | ошибок Excel: 0
SHA-256: 06f90aee9b2b6edf209f6a64ce662115b4a11ab4ca7ae285f380d817d3402ceb
Количество платежей: 334
payment_date  payment  interest  principal  closing_balance  source_row
  2022-09-22  26003.2  16482.09    9521.11       3636516.78          10
  2022-10-22  26003.2  22416.88    3586.32       3632930.46          11
  2022-11-22  26003.2  23141.27    2861.93       3630068.53          12
  2022-12-22  26003.2  22377.13    3626.07       3626442.46          13
  2023-01-22  26003.2  23099.94    2903.26       3623539.20          14


## 2. Типы, пропуски и уникальность
Один договор. Дата уникальна внутри графика, source_row указывает строку источника. Это ещё не банковский ключ «объект + дата оценки». Нулевой остаток не считаем пропуском.

In [3]:
profile = profile_table(frame, date_column="payment_date",
                        key_columns=["payment_date"], exclude=["source_row"])
print(pd.DataFrame(profile["fields"]).to_string(index=False))
print("Дубликаты строк:", profile["full_duplicate_rows"])
print("Повторные ключи:", profile["duplicate_keys"])
print("Строки с пустым ключом:", profile["missing_key_rows"])

          field   dtype  missing  missing_share  unique_non_null  candidate_factor  non_finite     iqr_lower    iqr_upper  iqr_flags
   payment_date  object        0            0.0              334             False         NaN           NaN          NaN        NaN
        payment float64        0            0.0                2              True         0.0   26003.20000 2.600320e+04        1.0
       interest float64        0            0.0              334              True         0.0   -4167.33625 3.523447e+04        0.0
      principal float64        0            0.0              334              True         0.0   -9231.27375 3.017054e+04        0.0
closing_balance float64        0            0.0              334              True         0.0 -714476.65625 5.679246e+06        0.0
     source_row   int64        0            0.0              334             False         0.0    -156.50000 5.095000e+02        0.0
Дубликаты строк: 0
Повторные ключи: 0
Строки с пустым ключом: 0


## 3. Распределения и выбросы
Квантили и частоты по интервалам показывают масштаб и форму распределений. Правило IQR: Q1 − 1,5×IQR, Q3 + 1,5×IQR. Это флаги для исследования, не ошибки. Почти постоянный платёж закономерно выделяет первый и последний платежи. Строки не удаляем.

In [4]:
numeric = ["payment", "interest", "principal", "closing_balance"]
print(frame[numeric].describe(percentiles=[.05, .25, .5, .75, .95]).to_string())
for column in numeric:
    print("\nРаспределение:", column)
    print(pd.cut(frame[column], bins=6).value_counts(sort=False).to_string())
    field = next(f for f in profile["fields"] if f["field"] == column)
    flagged = frame[(frame[column] < field["iqr_lower"]) | (frame[column] > field["iqr_upper"])]
    print("IQR-флаги:", len(flagged))
    if len(flagged):
        print(flagged[["payment_date", column, "source_row"]].head(10).to_string(index=False))

            payment      interest     principal  closing_balance
count    334.000000    334.000000    334.000000     3.340000e+02
mean   26003.206766  15086.925659  10916.281108     2.404501e+06
std        0.123662   6342.435204   6342.451212     1.020304e+06
min    26003.200000    127.620000   2861.930000     0.000000e+00
5%     26003.200000   2760.234500   3506.996500     4.100842e+05
25%    26003.200000  10608.342500   5544.405000     1.683169e+06
50%    26003.200000  16765.490000   9237.710000     2.685296e+06
75%    26003.200000  20458.795000  15394.857500     3.281600e+06
95%    26003.200000  22496.203500  23242.965500     3.579727e+06
max    26005.460000  23141.270000  25877.840000     3.636517e+06

Распределение: payment
payment
(26003.198, 26003.577]    333
(26003.577, 26003.953]      0
(26003.953, 26004.33]       0
(26004.33, 26004.707]       0
(26004.707, 26005.083]      0
(26005.083, 26005.46]       1
IQR-флаги: 1
payment_date  payment  source_row
  2050-06-15 26005.46     

## 4. Временная структура
Проверяем порядок дат, интервалы и число строк в месяцах. Платежи 2022–2050 годов — будущий график на одной исходной дате. Они не составляют историю месячных наблюдений банковского ЭК.

In [5]:
temporal = profile["temporal"]
for key in ["start", "end", "chronological_in_input", "missing_dates",
            "invalid_non_null_dates", "repeated_dates", "gap_days_counts"]:
    print(key, ":", temporal[key])
print("Число месяцев:", len(temporal["rows_per_month"]))
print("Распределение числа строк в месяце:", pd.Series(temporal["rows_per_month"]).value_counts().to_dict())

start : 2022-09-22
end : 2050-06-15
chronological_in_input : True
missing_dates : 0
invalid_non_null_dates : 0
repeated_dates : 0
gap_days_counts : {'24.0': 1, '28.0': 21, '29.0': 7, '30.0': 111, '31.0': 193}
Число месяцев: 334
Распределение числа строк в месяце: {1: 334}


## 5. Связи признаков и числовые сверки
Корреляция Спирмена описывает связи в графике. payment = interest + principal и движение долга создают механические зависимости. Нельзя автоматически считать части потока независимыми причинами изменения ЭК. Категориальных факторов и нескольких договоров здесь нет.

In [6]:
print(frame[numeric].corr(method="spearman").round(4).to_string())
checks = pd.DataFrame(audit["checks"])
print("\nСверки:")
print(checks.groupby("check")["passed"].agg(["count", "sum"]).to_string())
print("Всего:", len(checks), "| прошли:", int(checks.passed.sum()))
assert audit["checks_passed"], "Есть расхождение в графике"
print("Вывод: обязательные поля, даты и выбранные формулы графика проверены.")
print("История банковского target и рыночные кривые отсутствуют.")

                 payment  interest  principal  closing_balance
payment           1.0000   -0.0946     0.0946          -0.0946
interest         -0.0946    1.0000    -1.0000           0.9885
principal         0.0946   -1.0000     1.0000          -0.9885
closing_balance  -0.0946    0.9885    -0.9885           1.0000

Сверки:
                        count  sum
check                             
annuity_formula             1    1
chronological_date        334  334
coefficient_recurrence    334  334
final_balance               1    1
first_date                  1    1
interest_formula          334  334
interest_total              1    1
last_date                   1    1
payment_split             334  334
payment_total               1    1
principal_rollforward     334  334
principal_total             1    1
Всего: 1677 | прошли: 1677
Вывод: обязательные поля, даты и выбранные формулы графика проверены.
История банковского target и рыночные кривые отсутствуют.


## 6. Изменение процентов сентября и октября
Сохранённые значения процентов предоставленного графика. Формула: остаток × договорная ставка × дни / 365. Shapley усредняет эффект замены фактора по двум порядкам. Остаток связан с округлением сохранённых процентов. Этот target — проценты графика, отдельно от NPV, dNPV и ЭК.

In [7]:
schedule = run_schedule_change(audit)
print("Даты:", schedule["dates"], "| интервалы:", schedule["days"])
print(f"Проценты: {schedule['recorded_before']:.2f} -> {schedule['recorded_after']:.2f}")
print(f"Изменение: {schedule['recorded_change']:+.6f}")
for factor, value in schedule["contributions"].items():
    print(f"Вклад {factor}: {value:+.6f}")
print(f"Остаток: {schedule['residual']:+.6f}")
assert np.isclose(schedule["recorded_change"], sum(schedule["contributions"].values()) + schedule["residual"], atol=1e-8)

Даты: ['2022-09-22', '2022-10-22'] | интервалы: [22, 30]
Проценты: 16482.09 -> 22416.88
Изменение: +5934.790000
Вклад balance: -50.866204
Вклад days: +5985.661373
Остаток: -0.005168


## 7. Учебное месячное изменение сценарного ЭК
**Синтетические условия на предоставленных потоках:** один договор, PV оставшихся положительных платежей, непрерывное дисконтирование ACT/365, ставка 8% на 31.08.2022 и 9% на 30.09.2022, стрессы ±200 б.п. Нет обязательств, начального потока, комиссий и опционов. Это NPV_demo, не банковский калькулятор.

dNPV_s = NPV_stress_s − NPV_base, EC_signed = min_s dNPV_s. Месячная разность — значение после минус до. Во всех смешанных состояниях пересчитываем базу, все стрессы и минимум. Время совместно учитывает истёкший платёж и сокращение оставшихся сроков.

In [8]:
demo = run_cashflow_demo(audit)
print("Даты:", demo["dates"])
for state in ["before", "after"]:
    print(f"ЭК_demo {state}: {demo[state]:+.6f}")
print(f"Месячная разность: {demo['change']:+.6f}")
for factor, value in demo["contributions"].items():
    print(f"Вклад {factor}: {value:+.6f}")
print(f"Остаток относительно функции опыта: {demo['residual']:+.6f}")
print("Взаимодействия (не добавляем повторно):", demo["interactions"])
assert np.isclose(demo["change"], sum(demo["contributions"].values()) + demo["residual"], atol=1e-8)
assert all(not pair["add_to_total"] for pair in demo["interactions"])
print("Точный баланс учебной функции не подтверждает качество на банковской истории.")

Даты: ['2022-08-31', '2022-09-30']
ЭК_demo before: -552373.725668
ЭК_demo after: -479166.568393
Месячная разность: +73207.157275
Вклад elapsed_days: +1037.849261
Вклад base_rate: +72169.308015
Остаток относительно функции опыта: +0.000000
Взаимодействия (не добавляем повторно): [{'factors': ['elapsed_days', 'base_rate'], 'index': -286.8049304257147, 'add_to_total': False}]
Точный баланс учебной функции не подтверждает качество на банковской истории.


## 8. Задание организаторов: ANN, SIMPLE и RGM
Организаторы предложили продисконтировать исходный ANN и рассчитать SIMPLE (тело в конце срока) и RGM (дифференцированный график). Исходный ANN сохраняем. Альтернативы рассчитываем на том же остатке 3 646 037,89 на 31.08.2022, тех же 334 датах и договорной ставке 7,5%.

Учебные правила: SIMPLE выплачивает проценты на остаток по каждой дате и всё тело в последний платёж. RGM погашает равную часть тела до копеек, последняя часть корректирует округление. Проценты: остаток начала интервала × 7,5% × фактические дни / 365. Не восстанавливаем потоки от даты открытия в 2020 году. Отдельная банковская спецификация типов не предоставлена.

In [9]:
from factor_analysis.payment_types import compare_payment_types, discount_schedule
comparison = compare_payment_types(audit, rates=(.075, .08, .09))
summary_rows = []
for item in comparison["summary"]:
    summary_rows.append({"type": item["payment_type"], "interest_total": item["total_interest"],
                         "payments_total": item["total_payment"], "last_payment": item["last_payment"],
                         **{f"PV_{float(rate)*100:g}%": value for rate, value in item["pv_by_rate"].items()}})
print(pd.DataFrame(summary_rows).round(2).to_string(index=False))
for kind, rows in comparison["schedules"].items():
    print("\n", kind, "первые 2 и последние 2 платежа:")
    print(pd.DataFrame(rows).iloc[[0, 1, -2, -1]].to_string(index=False))
    assert np.isclose(sum(row["principal"] for row in rows), audit["parameters"]["principal"], atol=.005)
    assert rows[-1]["closing_balance"] == 0
    assert all(np.isclose(row["payment"], row["principal"]+row["interest"], atol=.005) for row in rows)

  type  interest_total  payments_total  last_payment    PV_7.5%      PV_8%      PV_9%
   ANN      5039033.17      8685071.06      26005.46 3638033.17 3473006.23 3177157.01
SIMPLE      7604236.54     11250274.43    3664018.35 3636072.72 3432647.68 3077665.99
   RGM      3812984.12      7459022.01      10970.49 3639429.43 3502461.10 3253453.98

 ANN первые 2 и последние 2 платежа:
payment_date  payment  interest  principal  closing_balance  source_row
  2022-09-22 26003.20  16482.09    9521.11       3636516.78          10
  2022-10-22 26003.20  22416.88    3586.32       3632930.46          11
  2050-05-22 26003.20    317.86   25685.34         25877.84         342
  2050-06-15 26005.46    127.62   25877.84             0.00         343

 SIMPLE первые 2 и последние 2 платежа:
payment_date    payment  interest  principal  closing_balance  opening_balance  days
  2022-09-22   16482.09  16482.09       0.00       3646037.89       3646037.89    22
  2022-10-22   22475.58  22475.58       0.00   

## 9. Дисконтирование и выводы сравнения
Рыночная ставка и правило дисконтирования не заданы. Поэтому ставки 7,5%, 8% и 9% являются **учебными условиями**, отдельно от договорных 7,5% для процентов. Используем непрерывное дисконтирование ACT/365: `DF_i = exp(−r × days_i/365)`, `PV_i = CF_i × DF_i`. Сумма PV относится к оставшимся положительным потокам. Нет начальной выдачи и обязательств, поэтому не называем результат банковской NPV.

SIMPLE дольше сохраняет весь долг и даёт большой последний платёж. При RGM тело возвращается раньше, чем при ANN/SIMPLE в этом опыте, и суммарные проценты меньше. Величина отдельного платежа зависит также от числа дней: дифференцированный график не обязан строго убывать на каждом соседнем интервале. Повышение ставки дисконтирования уменьшает PV всех трёх положительных графиков. В рассматриваемых учебных условиях PV RGM выше PV ANN, а PV ANN выше PV SIMPLE. Это вывод для данных параметров, не универсальное правило для любой методики.

In [10]:
discounted_ann = discount_schedule(comparison["schedules"]["ANN"], comparison["valuation_date"], .08)
print(pd.DataFrame(discounted_ann)[["payment_date", "payment", "days_from_valuation", "discount_factor", "pv"]].head().round(6).to_string(index=False))
for item in comparison["summary"]:
    pv = item["pv_by_rate"]
    assert pv["0.075"] > pv["0.08"] > pv["0.09"]
print("Вывод: проверены расчёт по каждой дате, суммирование PV и чувствительность к ставке.")
print("Учебные ставки и построенные SIMPLE/RGM не являются наблюдаемой банковской историей.")

payment_date  payment  days_from_valuation  discount_factor           pv
  2022-09-22  26003.2                   22         0.995190 25878.116520
  2022-10-22  26003.2                   52         0.988667 25708.517240
  2022-11-22  26003.2                   83         0.981973 25534.432273
  2022-12-22  26003.2                  113         0.975537 25367.085420
  2023-01-22  26003.2                  144         0.968931 25195.312455
Вывод: проверены расчёт по каждой дате, суммирование PV и чувствительность к ставке.
Учебные ставки и построенные SIMPLE/RGM не являются наблюдаемой банковской историей.


## 10. Первый месячный переход для разных типов
Сравниваем первые два начисления процентов. Target здесь — проценты, не NPV/dNPV/ЭК. Та же формула и Shapley перехода разделяют изменение числа дней и остатка долга. Для SIMPLE до последнего платежа остаток постоянный, поэтому в этом переходе вклад долга нулевой. Разные типы погашения дают разные траектории факторов.

In [11]:
for kind, rows in comparison["schedules"].items():
    local = run_schedule_change({**audit, "cashflows": rows})
    print(kind, "| изменение процентов:", round(local["recorded_change"], 6),
          "| дни:", round(local["contributions"]["days"], 6),
          "| долг:", round(local["contributions"]["balance"], 6),
          "| остаток:", round(local["residual"], 6))
    assert np.isclose(local["recorded_change"], sum(local["contributions"].values())+local["residual"], atol=1e-8)

ANN | изменение процентов: 5934.79 | дни: 5985.661373 | долг: -50.866204 | остаток: -0.005168
SIMPLE | изменение процентов: 5993.49 | дни: 5993.486942 | долг: 0.0 | остаток: 0.003058
RGM | изменение процентов: 5926.19 | дни: 5984.514658 | долг: -58.319852 | остаток: -0.004805


## 11. Структурированный результат и дальнейшая работа

Сохраняем профиль, исходный ANN, вычисленные SIMPLE/RGM, дисконтирование и месячные опыты отдельно. Исходная книга не меняется.

Первый спринт: изучили данные, разобрались в структуре платежей, сравнили три типа, выполнили учебное дисконтирование и попробовали месячное объяснение. Для банковского анализа нужны месячные снимки NPV/ЭК и факторов, область агрегации и версии методики. Глобальная важность банковских факторов и живая LLM пока не проверены.


In [ ]:
PROJECT_DIR = Path("/Users/andrey/Documents/Прога/Project_T1")
OUT = PROJECT_DIR / "data"
OUT.mkdir(parents=True, exist_ok=True)
result = {"source": audit["source"], "profile": profile, "schedule_change": schedule,
          "ec_demo": demo, "payment_types": comparison, "data_kind": "provided_schedule_and_synthetic_valuation_conditions"}
(OUT / "research_result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Сохранён исследовательский результат:", OUT / "research_result.json")
PAYMENT_OUT = PROJECT_DIR / "data"
PAYMENT_OUT.mkdir(parents=True, exist_ok=True)
(PAYMENT_OUT / "comparison.json").write_text(json.dumps(comparison, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
for kind, rows in comparison["schedules"].items():
    pd.DataFrame(rows).to_csv(PAYMENT_OUT / f"{kind}.csv", index=False)
print("Сохранены сравнение и три графика CSV:", PAYMENT_OUT)
